In [4]:
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# Create dataset
X, y = make_classification(n_classes=2, n_samples=1000, n_features=20, random_state=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1
)

# Base Models
dt = DecisionTreeClassifier(random_state=1)

xgb = XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=1
)

cat = CatBoostClassifier(
    verbose=0,
    random_state=1
)

# Optional: Tune CatBoost
param_grid = {
    'iterations': [50, 100],
    'learning_rate': [0.01, 0.1],
    'depth': [4, 6]
}

grid_cat = GridSearchCV(
    estimator=cat,
    param_grid=param_grid,
    cv=3,
    verbose=1
)

grid_cat.fit(X_train, y_train)

best_cat = grid_cat.best_estimator_

# Stacking Model
base_models = [
    ('dt', dt),
    ('xgb', xgb),
    ('cat', best_cat)
]

meta_model = LogisticRegression()

stack_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5
)

stack_model.fit(X_train, y_train)

y_pred = stack_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Fitting 3 folds for each of 8 candidates, totalling 24 fits


C:\Users\Jagdish singh\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:200: UserWarning: [08:36:46] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\Jagdish singh\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:200: UserWarning: [08:36:46] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\Jagdish singh\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:200: UserWarning: [08:36:46] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\Jagdish singh\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:200: UserWarning: [08:36:47] WARNING: C:\actions-runner\_w

Accuracy: 0.87
Confusion Matrix:
 [[124  15]
 [ 24 137]]
Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.89      0.86       139
           1       0.90      0.85      0.88       161

    accuracy                           0.87       300
   macro avg       0.87      0.87      0.87       300
weighted avg       0.87      0.87      0.87       300

